In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers

train_ds = keras.utils.image_dataset_from_directory(
    'mvtec_dataset/train',
    labels=None,
    image_size=(128,128),
    batch_size=32,
    shuffle=True
)


normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x: normalization_layer(x))

model = keras.Sequential([
    layers.Conv2D(32,(3,3),activation='relu',padding='same',input_shape=(28,28,3)),
    layers.MaxPooling2D((2,2),padding='same'),
    layers.Conv2D(16,(3,3),activation='relu',padding='same'),
    layers.MaxPooling2D((2,2),padding='same'),

    layers.Conv2D(16,(3,3),activation='relu',padding='same'),
    layers.UpSampling2D((2,2)),
    layers.Conv2D(32,(3,3),activation='relu',padding='same'),
    layers.UpSampling((2,2)),
    layers.Conv2D(3,(3,3),activation='sigmoid',padding='same')
    
])

model.compile(optimizer='adam', loss='mse')

H = model.fit(train_ds, epochs=15)

for normal_batch in train_ds.take(1):
    preds = model.predict(normal_batch)
    mse_errors = np.mean(np.power(normal_batch-preds,2),axis=(1,2,3))
    threshold = np.percentile(mse_errors, 99)

print(f"Calculated Anomaly Threshold: {threshold}")


test_ds = keras.utils.image_dataset_from_directory(
    'mvtec_dataset/test',
    image_size=(28,28),
    batch_size=32,
    label_mode='categorical'
)

test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

for images, labels in test_ds.take(1):
    test_preds=model.predict(images)
    test_mse = np.mean(np.power(images-test_preds,2), axis=(1,2,3))

    anomalies = test_mse > threshold
    print(f"Detected defective/anomalous images in batch: {sum(anomalies)} out of {len(images)}")